# MFA Training Hyperparameters

This notebook explains the knobs you need to train MFA at large scale. It is also runnable: the bundled hierarchy dataset is used as a small smoke test so every setting below connects to the actual pipeline.


In [1]:
from pathlib import Path
from types import SimpleNamespace
import torch

# Run this notebook from the repository root.
from demo_data.hierarchy_json import load_hierarchy_records, make_hierarchy_dataloaders
from large_scale_mfa_pipeline import make_loaders, run_from_config


## 1. Data Loader

The pipeline only assumes a PyTorch loader. Each batch should be either `x` or `(x, tokens)`, where `x` is `(batch, activation_dim)`.

For our large scale runs we replace `make_loaders_for_layer` with a streaming activation loader.


In [2]:
DATA_PATH = Path('data') / 'hierarchy_dataset_long.json'
records = load_hierarchy_records(DATA_PATH)
print(f'{len(records):,} demo rows')
records[0]


17,150 demo rows


{'parent': 'bird',
 'level': 'child',
 'concept': 'chat',
 'sentence': 'A brown chat searched for food on the forest floor.'}

In [3]:
FEATURE_DIM = 128       # demo feature dimension; real activations define this
BATCH_SIZE = 256        # per-process batch size
MAX_EXAMPLES = 2_000    # demo only
VAL_FRACTION = 0.2      # demo only; real validation should be a fixed small subset
SEED = 0
DATA_DTYPE = torch.float32


def make_loaders_for_layer(layer):
    train_loader, val_loader, _ = make_hierarchy_dataloaders(
        DATA_PATH,
        feature_dim=FEATURE_DIM,
        batch_size=BATCH_SIZE,
        val_fraction=VAL_FRACTION,
        max_examples=MAX_EXAMPLES,
        seed=SEED + int(layer),
        dtype=DATA_DTYPE,
    )
    return train_loader, val_loader

train_loader, val_loader = make_loaders_for_layer(0)
x, meta = next(iter(train_loader))
print('batch:', x.shape, 'metadata:', meta.shape)


batch: torch.Size([256, 128]) metadata: torch.Size([256])


## 2. Run Identity

These only affect naming and where outputs go.

- `OUTPUT_DIR`: checkpoints and centroids.
- `RUN_NAME`: readable name in logs and filenames.
- `LAYERS`: train one or more activation layers.


In [4]:
OUTPUT_DIR = Path('runs') / 'notebook_demo'
RUN_NAME = 'notebook_demo_mfa'
MODEL_NAME = 'hierarchy-hashed-demo'
LAYERS = [0]


## 3. MFA Size

These are the most important hyper-parameters to set. 
In our 100 million activation runs, we saw good performance with 8K.
In the end, if we want to isolate structures in single gaussians, then we use less gaussians, but if we want more accurate structures -- be able to model curved surfaces -- we need to tile them with multiple gaussians and post-hoc connect them via BFS (neighboring gaussians).

- `NUM_COMPONENTS` (`K`): number of local regions. Cost grows roughly linearly with K.
- `RANK`: local factor dimension in each component. Cost also grows with rank. We found rank 10 works decently, but some fine-grained features require higher rank. Also requires higher rank for reconstruction (ideally should be intrinsic dimension).


In [5]:
NUM_COMPONENTS = 25  # Value set just for this demolstration; real models may need more components to capture the structure of the data.
RANK = 4            # Value set just for this demonstration; real models may need higher rank to capture the structure of the data.


## 4. Runtime

- `single_gpu`: simple local run.
- `mgpu`: use Accelerate/FSDP for multi-GPU training.
- `DEVICE`: usually `cuda`; this smoke test uses CPU.


In [6]:
TRAINING_BACKEND = 'single_gpu'
DEVICE = 'cpu'


## 5. Centroid Initialization

Initialization is one of the most important decisions for proper convergence. 
For the large scale MFAs in the paper we used projected K-Means, but our current implementation is not efficient enough for very large scale training runs. In that case I suggest using random init:

Random init means we initialize centroids to random points, important: randomly sampled weights does not converge to a good minimum.

Important knobs for K-Means:

- `KMEANS_POOL_SIZE`: sampled activations used for K-Means. Bigger is better but costs I/O/memory.
- `PROJECTED_DIM`: random projection dimension for faster K-Means. Use 128-512; 256 is a good default.
- `KMEANS_REFINE_EPOCHS`: full-dimensional refinement passes. Use 1-3 at scale.



In [28]:
INIT_METHOD = 'projected_kmeans' # 'projected_kmeans' or 'random'
CENTROIDS_PATH = OUTPUT_DIR / 'centroids' / 'centroids_{model_name}_L{layer}_k{num_components}.pt'
FORCE_REINIT = False

KMEANS_POOL_SIZE = 512   # smoke test; large-scale: 500k-8M
PROJECTED_DIM = 32       # smoke test; large-scale default: 256
KMEANS_ITERS = 10        # smoke test; large-scale: 20-50
KMEANS_RESTARTS = 2      # smoke test; large-scale: 2-5
KMEANS_REFINE_EPOCHS = 2
KMEANS_TOL = 1e-4
KMEANS_METRIC = 'euclidean'

USE_TOKEN_WEIGHTS = False
MODEL_VOCAB_SIZE = 16
KMEANS_SMOOTHING = 1.0
KMEANS_POWER = 1.0


## 6. MFA Parameter Initialization

These are usually safe to leave alone.

- `PSI_INIT`: initial diagonal noise variance.
- `PSI_PER_COMPONENT`: separate noise vector per component. More flexible, more parameters.
- `SCALE_INIT`: initial loading scale.
- `EPS_FLOOR`: numerical stability floor.


In [29]:
PSI_INIT = 1.0
PSI_PER_COMPONENT = False
SCALE_INIT = 1.0
EPS_FLOOR = 1e-5


## 7. Optimization

For huge streams, do not define an epoch as a full data pass. Use `STEPS_PER_EPOCH` as a fixed budget.
For our large scale MFAs, we ran a single epoch of 100 million activations.

- `LR`: start near `7e-5` for large runs.
- `NUM_EPOCHS * STEPS_PER_EPOCH`: total training step budget.
- `VAL_INTERVAL`: MGPU validation/checkpoint cadence.


In [30]:
NUM_EPOCHS = 2          # smoke test; large-scale depends on step budget
STEPS_PER_EPOCH = None # smoke test; large-scale example: 50_000
LR = 1e-3              # smoke test; large-scale default: 7e-5
GRAD_CLIP = None
LOG_INTERVAL = 5
START_EPOCH = 1
VAL_INTERVAL = 100
WANDB_PROJECT = None
RESUME_CHECKPOINT = None


## 8. Build the Config

This object is exactly what `run_training.py --config ...` provides from a Python config file. Keeping it in the notebook makes the flow easy to inspect.


In [31]:
cfg = SimpleNamespace(
    OUTPUT_DIR=OUTPUT_DIR,
    RUN_NAME=RUN_NAME,
    MODEL_NAME=MODEL_NAME,
    LAYERS=LAYERS,
    NUM_COMPONENTS=NUM_COMPONENTS,
    RANK=RANK,
    TRAINING_BACKEND=TRAINING_BACKEND,
    DEVICE=DEVICE,
    DATA_DTYPE=DATA_DTYPE,
    SEED=SEED,
    make_loaders=make_loaders_for_layer,
    INIT_METHOD=INIT_METHOD,
    CENTROIDS_PATH=CENTROIDS_PATH,
    FORCE_REINIT=FORCE_REINIT,
    KMEANS_POOL_SIZE=KMEANS_POOL_SIZE,
    PROJECTED_DIM=PROJECTED_DIM,
    MODEL_VOCAB_SIZE=MODEL_VOCAB_SIZE,
    USE_TOKEN_WEIGHTS=USE_TOKEN_WEIGHTS,
    KMEANS_SMOOTHING=KMEANS_SMOOTHING,
    KMEANS_POWER=KMEANS_POWER,
    KMEANS_ITERS=KMEANS_ITERS,
    KMEANS_RESTARTS=KMEANS_RESTARTS,
    KMEANS_TOL=KMEANS_TOL,
    KMEANS_METRIC=KMEANS_METRIC,
    KMEANS_REFINE_EPOCHS=KMEANS_REFINE_EPOCHS,
    PSI_INIT=PSI_INIT,
    PSI_PER_COMPONENT=PSI_PER_COMPONENT,
    SCALE_INIT=SCALE_INIT,
    EPS_FLOOR=EPS_FLOOR,
    NUM_EPOCHS=NUM_EPOCHS,
    LR=LR,
    GRAD_CLIP=GRAD_CLIP,
    LOG_INTERVAL=LOG_INTERVAL,
    STEPS_PER_EPOCH=STEPS_PER_EPOCH,
    START_EPOCH=START_EPOCH,
    VAL_INTERVAL=VAL_INTERVAL,
    WANDB_PROJECT=WANDB_PROJECT,
    RESUME_CHECKPOINT=RESUME_CHECKPOINT,
)


## 9. Check or Run

The first cell only checks loader construction. Uncomment training when ready.


In [32]:
train_loader, val_loader = make_loaders(cfg, layer=0)
print(next(iter(train_loader))[0].shape)

# Uncomment to train the smoke-test MFA.
results = run_from_config(cfg)
results


torch.Size([256, 128])
[layer 0] preparing loaders
[init] running projected K-Means for 25 centroids
sampling for KNN
finished sampling
using projection
Running refinement epochs
[init] saved centroids to runs/notebook_demo/centroids/centroids_hierarchy-hashed-demo_L0_k25.pt
[layer 0] training with backend=single_gpu


Epoch 01 | Step 000005 Train NLL=119.374660: : 7it [00:00, 196.71it/s]


[epoch 01] train NLL=119.346904  val NLL=119.155816 ** best **


Epoch 02 | Step 000005 Train NLL=119.064479: : 7it [00:00, 203.46it/s]

[epoch 02] train NLL=119.034304  val NLL=118.840474 ** best **
Restored best model from epoch 02 with metric=118.840474
[layer 0] done. model path: runs/notebook_demo/models/notebook_demo_mfa_L0.ckpt


[{'layer': 0,
  'save_path': 'runs/notebook_demo/models/notebook_demo_mfa_L0.ckpt',
  'result': {'best_epoch': 2, 'best_metric': 118.84047393798828}}]

## Loading Models

Use the loader that matches how the checkpoint was written. Single-GPU training writes a plain MFA checkpoint through `modeling.mfa.save_mfa`; MGPU training writes an Accelerate/FSDP-friendly checkpoint through `modeling.model_checkpointing.save_mfa`.

Single-GPU checkpoints usually look like `runs/<run>/models/<run_name>_L<layer>.ckpt`. MGPU checkpoints are saved at validation intervals with the epoch prepended, for example `runs/<run>/models/2_<run_name>_L<layer>.ckpt`.

After loading, move the model to the device you want for analysis or inference and call `.eval()`.


In [33]:
from pathlib import Path
import torch

# Set this to the backend used for training: "single_gpu" or "mgpu".
trained_with = TRAINING_BACKEND

# Single GPU final checkpoint:
checkpoint_path = OUTPUT_DIR / "models" / f"{RUN_NAME}_L0.ckpt"

# MGPU checkpoints include the epoch prefix. Example:
# checkpoint_path = OUTPUT_DIR / "models" / f"1_{RUN_NAME}_L0.ckpt"

if trained_with == "mgpu":
    from modeling.model_checkpointing import load_mfa
else:
    from modeling.mfa import load_mfa

model = load_mfa(str(checkpoint_path), map_location="cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
model


MFA()

## Analysis

The github has all the code for visualization, steering and interpreting components:

https://github.com/ordavid-s/decomposing-activations-local-geometry
